# InfIndex Analysis
Inflammation index (InfIndex) and marker analysis for the PDP1 clinical trial.

In [1]:
### Imports
import src.config as config
import sys
sys.path.append(config.path_szb_commons)
import commons_codebase.src.core as commons
import src.folders as folders
import src.core as core
import pandas as pd
import os

import warnings # We save whether model converged or not in output
import statsmodels
warnings.filterwarnings("ignore", category=statsmodels.tools.sm_exceptions.ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Data Wrangling

In [2]:
### Create master dataframe with all experiemntal data
df_master = core.DataWrangl.get_df_master(
    exclude_pids=[1060], # Exclude patient 1060 (not one of the 12 PDP1 completers - only has mitokine data at baseline, no other markers)
    save={
        'dir_out': folders.exports, 
        'fname_out': 'infindex_master.csv'})
df_master.head(3)

Excluded 1 participant(s): [1060]


,trial,pID,tp,ndays,time,type,measure,is_risk,value,delta_value,log_value,delta_log_value
0,PDP1,1055,bsl,0.0,NaN,explore,eotaxin,False,146.0062,0.0000,4.9905,0.0000
1,PDP1,1055,A1,18.0,NaN,explore,eotaxin,False,185.0147,39.0085,5.2258,0.2354
2,PDP1,1055,B1,32.0,NaN,explore,eotaxin,False,190.3025,44.2964,5.2539,0.2634


In [ ]:
### Create table of observed means and SDs
df_master = pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv'))
df_observed = commons.Analysis.get_df_observed(
    df_master=df_master.rename(columns={'value': 'score'})) 
    # get_df_observed() calculates the mean and SD of the 'score' column

# Sort rows
df_observed['measure'] = pd.Categorical(
    df_observed['measure'], 
    categories=config.markers, 
    ordered=True)
df_observed = df_observed.sort_values(by=['measure', 'tp']).reset_index(drop=True)

# Format measure names with Greek letters
df_observed['measure'] = df_observed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')
df_observed.to_csv(os.path.join(folders.exports, 'infindex_observed.csv'), index=False)

In [ ]:
### Create table of log-tranformed means and SDs
df_master = pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv'))
df_observed = commons.Analysis.get_df_observed(
    df_master=df_master.rename(columns={'log_value': 'score'})) 

# Sort rows
df_observed['measure'] = pd.Categorical(
    df_observed['measure'], 
    categories=config.markers, 
    ordered=True)
df_observed = df_observed.sort_values(by=['measure', 'tp']).reset_index(drop=True)

# Format measure names with Greek letters
df_observed['measure'] = df_observed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')
df_observed.to_csv(os.path.join(folders.exports, 'infindex_observed_logtransformed.csv'), index=False)

## Statistical models of change

In [3]:
### Paired t-tests and Wilcoxon tests
df_pairedt_wilcoxon = core.Models.fit_paired_ttests(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    dir_out=folders.exports,
    fname_prefix='infindex_pairedt_wilcoxon')
df_pairedt_wilcoxon.head(3)

,measure,tp,smd,smd_ci_low,smd_ci_high,t,p_ttest,sig_ttest,p_ttest_adj,p_ttest_adj_sig,w,p_wilcoxon,sig_wilcoxon,p_wilcoxon_adj,p_wilcoxon_adj_sig
0,infindex,A1,-0.72,-1.58,0.14,3.128,0.017,*,0.050,*,3.0,0.039,*,0.078,
1,infindex,B1,-0.60,-1.43,0.23,2.413,0.047,*,0.070,,5.0,0.078,,0.078,
2,infindex,B30,-0.62,-1.46,0.22,1.986,0.082,,0.082,,7.0,0.074,,0.078,


In [6]:
### Mixed-effects models
df_mixedeffects = core.Models.fit_withinarm_septps(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')), 
    outcome='log_value',
    dir_out=folders.exports,
    fname_prefix='infindex_mixedeffects')
df_mixedeffects.head(3)

,measure,tp,did_converge,est,se,ci_low,ci_high,p,p_sig,p_adj,p_adj_sig
0,infindex,A1,True,-0.464,0.143,-0.744,-0.184,0.001,**,0.003,**
1,infindex,B1,True,-0.408,0.165,-0.732,-0.084,0.014,*,0.021,*
2,infindex,B30,True,-0.392,0.197,-0.779,-0.005,0.047,*,0.047,*


# Correlations

In [ ]:
### Correlation matrices of change between biomarkers
markers = config.markers_infind_comps + config.markers_mitokines 
df_master = pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv'))
df_master = df_master.rename(columns={'log_value': 'score'})

for tp in config.tps:
    df = commons.DataWrangl.widen_master(
        df_master=df_master,
        xvars=markers,
        x_tp=tp,
        x_use_delta=False,
        yvars=markers,
        y_tp=tp,
        y_use_delta=False)

    commons.Analysis.get_corrmats(
        df=df,
        xvars=markers,
        yvars=markers,
        draw=True,
        title=f'[1+ln(pg/mL)] correlation matrix at {tp}',
        dir_out=folders.corrmats,
        fname_out=f'infindex_corrmat_{tp}')

In [8]:
### Correlations between biomarkers and PD-related symptoms 
# Note: we are not sharing PD outcomes data publicly, so this cell wont run 

df_master = pd.read_csv(os.path.join(folders.exports, 'private', 'infindex_pdp1_master.csv'))
markers = ['infindex'] + config.markers_infind_comps + config.markers_mitokines
pd_symps = ['HAMA', 'MADRS', 'UPDRS_1', 'UPDRS_2', 'UPDRS_3', 'UPDRS_4', 'UPDRS_SUM',]

for tps in [('A1', 'A7'), ('B1', 'B7'), ('B30', 'B30')]:

    x_tp = tps[0]
    y_tp = tps[1]
    
    df = commons.DataWrangl.widen_master(
        df_master = df_master,
        xvars = markers,
        x_tp = x_tp,
        x_use_delta = True,
        yvars = pd_symps,
        y_tp = y_tp,
        y_use_delta = True)

    commons.Analysis.get_corrmats(
        df = df,
        xvars = markers,
        yvars = pd_symps,
        methods = ['spearman'],
        draw = True,
        title = f'(Δ log concentration @{x_tp}) vs. (Δ PD symptoms @{y_tp}) correlations',
        dir_out = os.path.join(folders.corrmats, 'markers vs PD symps'),
        fname_out = f'infindex_pdsymps_corrmat_{x_tp}_{y_tp}',)

## Visualizations

In [ ]:
### Combined (individual + group) concentration trajectories 
core.Plots.draw_conc_trajectory(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_combined')

In [ ]:
### Individual's concentration trajectories 
core.Plots.draw_conc_trajectory(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=False,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_inds')

In [ ]:
### Combined (individual + group) concentration trajectories with 95CI errorbars
core.Plots.draw_conc_trajectory_ci95(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_ci95')

In [ ]:
### Combined (individual + group) concentration trajectories with Morey (2008) within-subject error bars
# publication: https://www.tqmp.org/RegularArticles/vol04-2/p061/
core.Plots.draw_conc_trajectory_morey(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    draw_group=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_morey')

In [ ]:
### Trajectories of mixed-effects model CIs
core.Plots.draw_conc_trajectory_mixedeffects(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    outcome='log_value',
    draw_inds=True,
    dir_out=folders.conc_trajectories,
    fname_prefix='infindex_trajectory_mixedeffects')

In [ ]:
### Z-score heatmaps of change
core.Plots.draw_zscore_heatmap(
    df_master=pd.read_csv(os.path.join(folders.exports, 'infindex_master.csv')),
    dir_out=folders.heatmaps,
    fname_prefix='infindex_heatmap')

# Publication tables

In [ ]:
# Inflamation markers table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_inflamation.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_inflamation.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')
df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1.csv'), index=False)
df_mixed.head(3)

In [ ]:
# Exploratory markers table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_exploratory.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_exploratory.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')

df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1_exploratory.csv'), index=False)
df_mixed.head(3)

In [ ]:
# Mitokines table
df_mixed = pd.read_csv(os.path.join(folders.exports, 'infindex_mixedeffects_mitokines.csv'))
df_pairedt = pd.read_csv(os.path.join(folders.exports, 'infindex_pairedt_wilcoxon_mitokines.csv'))
df_pairedt = df_pairedt[['measure', 'tp', 'smd', 'smd_ci_low', 'smd_ci_high']]

df_mixed = pd.merge(df_mixed, df_pairedt, on=['measure', 'tp'], how='left')
df_mixed = df_mixed[[
    'measure', 'tp',
    'smd', 'smd_ci_low', 'smd_ci_high',
    'est', 'se', 'ci_low', 'ci_high', 
    'p', 'p_sig', 'p_adj', 'p_adj_sig']]

# Format measure names with Greek letters
df_mixed['measure'] = df_mixed['measure'].str.replace('TNF-alpha', 'TNF-α').str.replace('IFN-gamma', 'IFN-γ')

df_mixed.to_csv(os.path.join(folders.exports, 'infindex_Table1_mitokines.csv'), index=False)
df_mixed.head(3)